# Lab Day 10: Finetune Qwen3-VL-4B (LoRA) để định vị hư hại mặt đường
Đọc `README.md` và `GUIDE.md` trước. Notebook này chỉ là **khung**: bạn tự viết mã theo pseudo code trong GUIDE.

## 0. Môi trường (Colab T4)

In [ ]:
# Runtime → Change runtime type → T4 GPU
!nvidia-smi -L
!pip -q install -r requirements-colab.txt

In [ ]:
# Kaggle: dùng Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY). KHÔNG viết khoá vào notebook.
import os
from google.colab import userdata
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

## 1. Dữ liệu: tải và chia split

In [ ]:
!python scripts/prepare_data.py --kaggle

In [ ]:
# Kiểm tra split theo phiên quay: số ảnh/box từng split, không phiên nào nằm ở hai split,
# và quy đổi YOLO -> JSON 0–1000 -> pixel khớp box gốc (mã trong lab/data_check.py).
!python -m lab.data_check
# Vẽ 6 ảnh train kèm box gốc (pothole đỏ, crack vàng, manhole xanh) -> report/figures/gt_samples_train.jpg
!python -m lab.viz gt --split train --n 6 --seed 10 --out report/figures/gt_samples_train.jpg
from IPython.display import Image as _Img
_Img("report/figures/gt_samples_train.jpg", width=900)

## 2. Nhãn: YOLO → JSON thang 0–1000

In [ ]:
# yolo_to_box(), objects_to_answer() và PROMPT nằm trong lab/common.py (dùng chung cho train/infer/GRPO).
import json
from lab.common import PROMPT, objects_to_answer, yolo_to_box, box_to_pixel, read_jsonl

ex = read_jsonl("data/splits/train.jsonl")[0]
ans = objects_to_answer(ex["objects"])        # JSON gọn, box sắp theo (y1, x1)
print(PROMPT)
print(ans)
# quy ngược về pixel để so với box YOLO gốc
for b in json.loads(ans)[:3]:
    print(b["label"], b["bbox_2d"], "->", [round(v, 1) for v in box_to_pixel(b["bbox_2d"], 640, 360)])

## 3. E0: zero-shot

In [ ]:
# YOUR CODE HERE: nạp model 4-bit, suy luận trên test → work/e0.jsonl (GUIDE bước 3)

In [ ]:
!python scripts/score.py --pred work/e0.jsonl --name E0 --out report/evidence/e0.json

## 4. Dựng dữ liệu huấn luyện: chat template và mask loss

In [ ]:
# YOUR CODE HERE: encode_example() và collate() (GUIDE bước 4)

## 5. Setup LoRA / đóng băng cho E1, E2, E3

In [ ]:
# YOUR CODE HERE: apply_setup(model, setup) (GUIDE bước 5)

## 6. Vòng huấn luyện

In [ ]:
# YOUR CODE HERE: vòng train, lưu adapter + run.json (GUIDE bước 6)

## 7. Suy luận và chấm E1–E3

In [ ]:
# YOUR CODE HERE: suy luận test → work/eN.jsonl rồi chạy scripts/score.py với --meta (GUIDE bước 7)

## 8. E4 (ablation tự chọn) và E5 (tuỳ chọn, GRPO)

In [ ]:
# YOUR CODE HERE: GUIDE bước 8 và 9